Author: Sanjay Nayak

Affliation: Department of Computer Science & Engineering, Texas A&M University

Copyright 2023 Sanjay Nayak

In [1]:
import datetime
import numpy as np
import pandas as pd
import seaborn
from collections import defaultdict

import torch
import torch.nn as nn
from torch.autograd import Variable

from sklearn import svm
from sklearn.utils import shuffle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit

import xgboost as xgb

import plotly.express as px
import matplotlib.pyplot as plt

## Notebook Execution Order Guide

Run the cells in this order for proper execution:
1. Import libraries (top section)
2. Load and process data (intent mapping, feature engineering)
3. Create train/test split
4. **Ollama Setup** (import ollama, intent_mapping)
5. **Define all functions** (divide_train_test, get_accuracy, create_few_shot_prompt, extract_intent_from_response, convert_features_to_natural_language)
6. Extract few-shot examples from X_train
7. Test Ollama with a few examples
8. Run batch prediction on all X_test
9. Calculate accuracy metrics

All the necessary code is included below in the correct order.

# Intent Prediction
1. Shopping
2. Recreation
3. Work & Travel
4. Studying
5. Chilling

# Load all data

In [2]:
clus_en_proc = pd.read_csv("refined_data.csv")

In [3]:
clus_en_proc.head()

,serial_number,user,charging_status,charging_status.1,Charging_type,bluetooth_status,bluetooth_type,timestamp_long,datetime,location,cluster,daytime,Mapping,hour,minute,day_of_week,is_weekend,semantic_location,Intent
0,0,user_1,Disharging,Disharging,1,On,1,1.660000e+12,09-09-2022 12:00,"(17.4684131, 78.5714633)",0,A,H,12,0,4,0,home,leisure
1,1,user_1,Disharging,Disharging,1,On,1,1.660000e+12,09-09-2022 12:00,"(17.4684131, 78.5714633)",0,A,H,12,0,4,0,home,leisure
2,2,user_1,Disharging,Disharging,1,On,1,1.660000e+12,09-09-2022 12:00,"(17.4684131, 78.5714633)",0,A,H,12,0,4,0,home,leisure
3,3,user_1,Disharging,Disharging,1,On,1,1.660000e+12,09-09-2022 12:00,"(17.4684131, 78.5714633)",0,A,H,12,0,4,0,home,leisure
4,4,user_1,Disharging,Disharging,1,On,1,1.660000e+12,09-09-2022 12:00,"(17.4682466, 78.5713394)",0,A,H,12,0,4,0,home,leisure


# Intent addition
Add intent column to the dataset by mapping the cluster to the 5 intents currently present.

Intent - 
1. leisure
2. study
3. work
4. travel
5. shopping
6. praying
7. wellness
8. other

Mapping Values	
* home - 1
* other - 8
* hotel - 1
* shop - 5
* school - 2
* bank - 3
* church - 6
* university - 2
* cafe - 1 
* airport - 4
* hospital - 7

In [4]:
clus_en_proc["semantic_location"].unique()

array(['home', 'other', 'hotel', 'shop', 'school', 'bank', 'church',
       'university', 'cafe', 'airport', 'hospital'], dtype=object)

In [5]:
def map_intent(x):
    # Map semantic location names to intents
    map_values = {
        'home': 1,      # leisure
        'other': 8,     # other
        'hotel': 1,     # leisure
        'shop': 5,      # shopping
        'school': 2,    # study
        'bank': 3,      # work
        'church': 6,    # praying
        'university': 2, # study
        'cafe': 1,      # leisure
        'airport': 4,   # travel
        'hospital': 7   # wellness
    }
    return map_values.get(x, 8)  # Default to 'other' if not found

In [6]:
clus_en_proc["Intent"] = clus_en_proc["semantic_location"].apply(lambda x: map_intent(x))

In [7]:
clus_en_proc.head()

,serial_number,user,charging_status,charging_status.1,Charging_type,bluetooth_status,bluetooth_type,timestamp_long,datetime,location,cluster,daytime,Mapping,hour,minute,day_of_week,is_weekend,semantic_location,Intent
0,0,user_1,Disharging,Disharging,1,On,1,1.660000e+12,09-09-2022 12:00,"(17.4684131, 78.5714633)",0,A,H,12,0,4,0,home,1
1,1,user_1,Disharging,Disharging,1,On,1,1.660000e+12,09-09-2022 12:00,"(17.4684131, 78.5714633)",0,A,H,12,0,4,0,home,1
2,2,user_1,Disharging,Disharging,1,On,1,1.660000e+12,09-09-2022 12:00,"(17.4684131, 78.5714633)",0,A,H,12,0,4,0,home,1
3,3,user_1,Disharging,Disharging,1,On,1,1.660000e+12,09-09-2022 12:00,"(17.4684131, 78.5714633)",0,A,H,12,0,4,0,home,1
4,4,user_1,Disharging,Disharging,1,On,1,1.660000e+12,09-09-2022 12:00,"(17.4682466, 78.5713394)",0,A,H,12,0,4,0,home,1


In [8]:
clus_df = clus_en_proc.drop(columns=["charging_status", "timestamp_long", "datetime", "Mapping", "daytime"])

In [9]:
clus_df.sample(5)

,serial_number,user,charging_status.1,Charging_type,bluetooth_status,bluetooth_type,location,cluster,hour,minute,day_of_week,is_weekend,semantic_location,Intent
127614,127614,user_5,Disharging,1,On,1,"(30.5904452, -96.3424473)",163,10,9,5,1,other,8
131032,131032,user_7,Disharging,1,Off,2,"(39.1369154, -76.6245959)",175,21,0,4,0,other,8
123146,123146,user_3,Charging,2,Off,2,"(12.9685973, 77.7157969)",159,19,0,2,0,other,8
98556,98556,user_16,Disharging,1,Off,2,"(30.630802, -96.3590325)",84,13,52,2,0,other,8
115,115,user_1,Disharging,1,Off,2,"(17.468415, 78.5714584)",0,15,0,4,0,home,1


In [10]:
clus_df.shape

(138164, 14)

In [11]:
users_unique = clus_df["user"].unique()

In [12]:
clus_df.select_dtypes(include=[np.number]).corr()

,serial_number,Charging_type,cluster,hour,minute,day_of_week,is_weekend,Intent
serial_number,1.000000,-0.017710,0.974884,0.014549,0.080842,-0.047838,-0.040823,0.277407
Charging_type,-0.017710,1.000000,-0.017941,-0.067336,-0.020792,0.019325,0.018063,0.068055
cluster,0.974884,-0.017941,1.000000,0.021696,0.070972,-0.037018,-0.041902,0.261745
hour,0.014549,-0.067336,0.021696,1.000000,-0.063801,-0.031373,-0.057903,0.012285
minute,0.080842,-0.020792,0.070972,-0.063801,1.000000,-0.009930,-0.012426,0.069800
day_of_week,-0.047838,0.019325,-0.037018,-0.031373,-0.009930,1.000000,0.757217,0.020513
is_weekend,-0.040823,0.018063,-0.041902,-0.057903,-0.012426,0.757217,1.000000,-0.008464
Intent,0.277407,0.068055,0.261745,0.012285,0.069800,0.020513,-0.008464,1.000000


### Correlation matrix of the features - hour, weekday, charging_type, bluetooth, wifi, charging_status, clusters, daytype

In [13]:
def change_user_to_int(df):
    df["users"] = df["user"].apply(lambda x: int(x.split("_")[1]))
    df.drop(columns=["user"], inplace=True)
    df.rename(columns={"users": "user"}, inplace=True)
    return df

In [14]:
"""
loc_hr_df = pd.read_csv("/content/drive/MyDrive/thesis/preprocessing/probabilities/location_hour_proba.csv")
loc_hr_df.drop(columns=["Unnamed: 0"], inplace=True)
print(f"Shape of location hour df probability: {loc_hr_df.shape}")

usr_loc_hr_df = pd.read_csv("/content/drive/MyDrive/thesis/preprocessing/probabilities/user_location_hour_proba.csv")
usr_loc_hr_df.drop(columns=["Unnamed: 0"], inplace=True)
usr_loc_hr_df = change_user_to_int(usr_loc_hr_df)
print(f"Shape of location hour df probability: {usr_loc_hr_df.shape}")

usr_loc_dy_df = pd.read_csv("/content/drive/MyDrive/thesis/preprocessing/probabilities/user_location_day_proba.csv")
usr_loc_dy_df.drop(columns=["Unnamed: 0"], inplace=True)
usr_loc_dy_df = change_user_to_int(usr_loc_dy_df)
print(f"Shape of location hour df probability: {usr_loc_dy_df.shape}")

usr_loc_dyt_df = pd.read_csv("/content/drive/MyDrive/thesis/preprocessing/probabilities/user_location_daytype_proba.csv")
usr_loc_dyt_df.drop(columns=["Unnamed: 0"], inplace=True)
usr_loc_dyt_df = change_user_to_int(usr_loc_dyt_df)
print(f"Shape of location hour df probability: {usr_loc_dyt_df.shape}")
"""

'\nloc_hr_df = pd.read_csv("/content/drive/MyDrive/thesis/preprocessing/probabilities/location_hour_proba.csv")\nloc_hr_df.drop(columns=["Unnamed: 0"], inplace=True)\nprint(f"Shape of location hour df probability: {loc_hr_df.shape}")\n\nusr_loc_hr_df = pd.read_csv("/content/drive/MyDrive/thesis/preprocessing/probabilities/user_location_hour_proba.csv")\nusr_loc_hr_df.drop(columns=["Unnamed: 0"], inplace=True)\nusr_loc_hr_df = change_user_to_int(usr_loc_hr_df)\nprint(f"Shape of location hour df probability: {usr_loc_hr_df.shape}")\n\nusr_loc_dy_df = pd.read_csv("/content/drive/MyDrive/thesis/preprocessing/probabilities/user_location_day_proba.csv")\nusr_loc_dy_df.drop(columns=["Unnamed: 0"], inplace=True)\nusr_loc_dy_df = change_user_to_int(usr_loc_dy_df)\nprint(f"Shape of location hour df probability: {usr_loc_dy_df.shape}")\n\nusr_loc_dyt_df = pd.read_csv("/content/drive/MyDrive/thesis/preprocessing/probabilities/user_location_daytype_proba.csv")\nusr_loc_dyt_df.drop(columns=["Unnamed

# Division of data into X and y

* Divide the dataset into inputs (X) and labels (y)

#### Get the start and end index of each user data

In [15]:
user_index = dict()
for usr in users_unique:
    tmp_x = clus_df[clus_df["user"]==usr]
    indxs = tmp_x.index
    user_index[usr] = {"start": indxs[0], "end": indxs[-1]}
print("Done")

Done


In [16]:
user_index

{'user_1': {'start': np.int64(0), 'end': np.int64(1099)},
 'user_10': {'start': np.int64(1100), 'end': np.int64(6960)},
 'user_11': {'start': np.int64(6961), 'end': np.int64(11744)},
 'user_12': {'start': np.int64(11745), 'end': np.int64(19723)},
 'user_13': {'start': np.int64(19724), 'end': np.int64(48661)},
 'user_14': {'start': np.int64(48662), 'end': np.int64(78073)},
 'user_15': {'start': np.int64(78074), 'end': np.int64(79123)},
 'user_16': {'start': np.int64(79124), 'end': np.int64(99906)},
 'user_18': {'start': np.int64(99907), 'end': np.int64(109591)},
 'user_19': {'start': np.int64(109592), 'end': np.int64(112081)},
 'user_2': {'start': np.int64(112082), 'end': np.int64(112808)},
 'user_20': {'start': np.int64(112809), 'end': np.int64(113859)},
 'user_21': {'start': np.int64(113860), 'end': np.int64(119424)},
 'user_22': {'start': np.int64(119425), 'end': np.int64(119780)},
 'user_23': {'start': np.int64(119781), 'end': np.int64(122934)},
 'user_3': {'start': np.int64(122935)

#### Labels of the dataset without onehot encoding

In [51]:
labels = clus_df["semantic_location"]
intent = clus_df["Intent"]
features_df = clus_df[["user", "hour", "day_of_week", "Charging_type", "bluetooth_type", "semantic_location"]]

#### One hot encoding of input data

In [52]:
clus_df.head(3)

,serial_number,user,charging_status.1,Charging_type,bluetooth_status,bluetooth_type,location,cluster,hour,minute,day_of_week,is_weekend,semantic_location,Intent
0,0,user_1,Disharging,1,On,1,"(17.4684131, 78.5714633)",0,12,0,4,0,home,1
1,1,user_1,Disharging,1,On,1,"(17.4684131, 78.5714633)",0,12,0,4,0,home,1
2,2,user_1,Disharging,1,On,1,"(17.4684131, 78.5714633)",0,12,0,4,0,home,1


#### Use the `user` column for model trainings

In [53]:
# onehot_df = pd.get_dummies(data=clust_en_df, columns=["user", "daytime", "charging_type", "wifi", "bluetooth", "charging_status"])
# onehot_df_all = pd.get_dummies(data=clust_en_df, columns=["user", "daytime", "charging_type", "wifi", "bluetooth", "charging_status", "clusters"])

onehot_features = pd.get_dummies(data=features_df, columns=features_df.columns)

In [54]:
onehot_features.head()

,user_user_1,user_user_10,user_user_11,user_user_12,user_user_13,user_user_14,user_user_15,user_user_16,user_user_18,user_user_19,...,semantic_location_bank,semantic_location_cafe,semantic_location_church,semantic_location_home,semantic_location_hospital,semantic_location_hotel,semantic_location_other,semantic_location_school,semantic_location_shop,semantic_location_university
0,True,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2,True,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
3,True,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
4,True,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False


In [55]:
onehot_features.columns

Index(['user_user_1', 'user_user_10', 'user_user_11', 'user_user_12',
       'user_user_13', 'user_user_14', 'user_user_15', 'user_user_16',
       'user_user_18', 'user_user_19', 'user_user_2', 'user_user_20',
       'user_user_21', 'user_user_22', 'user_user_23', 'user_user_3',
       'user_user_5', 'user_user_6', 'user_user_7', 'user_user_8',
       'user_user_9', 'hour_0', 'hour_1', 'hour_2', 'hour_3', 'hour_4',
       'hour_5', 'hour_6', 'hour_7', 'hour_8', 'hour_9', 'hour_10', 'hour_11',
       'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17',
       'hour_18', 'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23',
       'day_of_week_0', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3',
       'day_of_week_4', 'day_of_week_5', 'day_of_week_6', 'Charging_type_1',
       'Charging_type_2', 'bluetooth_type_1',
       'bluetooth_type_1: Device C1nected', 'bluetooth_type_2',
       'semantic_location_airport', 'semantic_location_bank',
       'semantic_location_cafe'

### Save the one hot features df and the labels df

In [56]:
labels.to_csv("labels.csv")
intent.to_csv("intents.csv")
onehot_features.to_csv("onehot_features.csv")

#### Remove `user` column from the dataframe and then train the models - not used

In [23]:
# clust_en_df = clust_en_df.drop(columns=["user"])

In [24]:
# onehot_df = pd.get_dummies(data=clust_en_df, columns=["daytime", "charging_type", "wifi", "bluetooth", "charging_status"])
# onehot_df_all = pd.get_dummies(data=clust_en_df, columns=["daytime", "charging_type", "wifi", "bluetooth", "charging_status", "clusters"])

In [25]:
# onehot_df.iloc[0]

In [26]:
# onehot_df_all.iloc[0]

In [27]:
# # columns to use. Use all columns except the cluster column
# X = onehot_df.drop(columns=["clusters"]).to_numpy()

In [28]:
# X[0]

In [29]:
# labels[0]

In [30]:
# y = labels
# y_onehot = onehot_df_all[["clusters_1", "clusters_2", "clusters_3", "clusters_4", "clusters_5", "clusters_6", "clusters_7"]].to_numpy()

In [31]:
# y_onehot.shape

#### Create numpy array of X and y

In [57]:
X_main = onehot_features.to_numpy()
X_main.shape

(138164, 68)

Use `labels` for location prediction

Use `intent` for intent prediction

In [58]:
# y_main = labels.to_numpy()
y_main = intent.to_numpy()
y_main.shape

(138164,)

In [59]:
# Take training and testing data from each user
def divide_train_test_user(X, y, ratio=0.7):
    X_train, X_test, y_train, y_test = None, None, None, None
    for us, idxs in user_index.items():
        divide = int(ratio*(idxs["end"] - idxs["start"]))

        if X_train is None:
            X_train = X[idxs["start"]: divide + idxs["start"]]
        elif X_train is not None:
            X_train = np.concatenate((X_train, X[idxs["start"]: divide + idxs["start"]]), axis=0)

        if y_train is None:
            y_train = y[idxs["start"]: divide + idxs["start"]]
        elif y_train is not None:
            y_train = np.concatenate((y_train, y[idxs["start"]: divide + idxs["start"]]), axis=0)
        
        if X_test is None:
            X_test = X[divide + idxs["start"]: idxs["end"] + 1]
        elif X_test is not None:
            X_test = np.concatenate((X_test, X[divide + idxs["start"]: idxs["end"]]), axis=0)
        
        if y_test is None:
            y_test = y[divide + idxs["start"]: idxs["end"] + 1]
        elif y_test is not None:
            y_test = np.concatenate((y_test, y[divide + idxs["start"]: idxs["end"]]), axis=0)
        
        # print(us, idxs["start"], divide + idxs["start"], idxs["end"] + 1, idxs["end"] - idxs["start"] + 1)
    
    return X_train, X_test, y_train, y_test

In [60]:
def divide_train_test(X, y, ratio=0.8):
    divide = int(ratio*X.shape[0])
    X_train, X_test = X[:divide], X[divide:]
    y_train, y_test = y[:divide], y[divide:]

    return X_train, X_test, y_train, y_test

#### Prediction function

In [61]:
def get_accuracy(y_pred, y):
    correct_pred = 0
    correct_pred += (y_pred == y).sum()/y.shape[0]

    return correct_pred

In [62]:
# Divide the data into train and test sets
X_train, X_test, y_train, y_test = divide_train_test_user(X_main, y_main, ratio=0.8)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (110507, 68)
X_test shape: (27637, 68)
y_train shape: (110507,)
y_test shape: (27637,)


In [63]:
X_train

array([[ True, False, False, ..., False, False, False],
       [ True, False, False, ..., False, False, False],
       [ True, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]],
      shape=(110507, 68))

In [64]:
def create_few_shot_prompt(examples, query):
    """
    Create a few-shot learning prompt for Ollama.
    examples: list of example strings
    query: the query/test case
    """
    prompt = """You are an expert at predicting user intents based on their sensor data and time information.
You will be given examples of user behavior patterns and their corresponding intents.
Then you will be given a new scenario and must predict only the intent (the number from 1-8).

Intent mapping:
1 = leisure
2 = study  
3 = work
4 = travel
5 = shopping
6 = praying
7 = wellness
8 = other

IMPORTANT: Output ONLY a single number (1, 2, 3, 4, 5, 6, 7, or 8) - nothing else.

Here are examples:

"""
    
    for i, example in enumerate(examples, 1):
        prompt += f"Example {i}:\n{example}\n\n"
    
    prompt += f"Now predict the intent for this scenario:\n{query}\n\nOutput only the intent number (1-8):"
    
    return prompt

print("Prompt creation function defined")

Prompt creation function defined


In [66]:
import ollama
import random

# Create mapping for intent numbers to human-readable names
intent_mapping = {
    1: "leisure",
    2: "study",
    3: "work",
    4: "travel",
    5: "shopping",
    6: "praying",
    7: "wellness",
    8: "other"
}

print("Ollama package imported successfully")

Ollama package imported successfully


In [67]:
# First, let's understand the feature columns (excluding user columns)
print("All feature columns in onehot_features:")
print(onehot_features.columns.tolist())
print(f"\nTotal features: {len(onehot_features.columns)}")
print(f"\nX_train shape: {X_train.shape}")
print(f"Y_train shape: {y_train.shape}")

All feature columns in onehot_features:
['user_user_1', 'user_user_10', 'user_user_11', 'user_user_12', 'user_user_13', 'user_user_14', 'user_user_15', 'user_user_16', 'user_user_18', 'user_user_19', 'user_user_2', 'user_user_20', 'user_user_21', 'user_user_22', 'user_user_23', 'user_user_3', 'user_user_5', 'user_user_6', 'user_user_7', 'user_user_8', 'user_user_9', 'hour_0', 'hour_1', 'hour_2', 'hour_3', 'hour_4', 'hour_5', 'hour_6', 'hour_7', 'hour_8', 'hour_9', 'hour_10', 'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18', 'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23', 'day_of_week_0', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3', 'day_of_week_4', 'day_of_week_5', 'day_of_week_6', 'Charging_type_1', 'Charging_type_2', 'bluetooth_type_1', 'bluetooth_type_1: Device C1nected', 'bluetooth_type_2', 'semantic_location_airport', 'semantic_location_bank', 'semantic_location_cafe', 'semantic_location_church', 'semantic_location_home', 'semanti

## OLLAMA FEW-SHOT LEARNING PIPELINE

All cells from here onwards are for Ollama integration. Run them in the order they appear.

In [68]:
def create_few_shot_prompt(examples, query):
    """
    Create a few-shot learning prompt for Ollama.
    examples: list of example strings
    query: the query/test case
    """
    prompt = """You are an expert at predicting user intents based on their sensor data and time information.
You will be given examples of user behavior patterns and their corresponding intents.
Then you will be given a new scenario and must predict only the intent (the number from 1-8).

Intent mapping:
1 = leisure
2 = study  
3 = work
4 = travel
5 = shopping
6 = praying
7 = wellness
8 = other

IMPORTANT: Output ONLY a single number (1, 2, 3, 4, 5, 6, 7, or 8) - nothing else.

Here are examples:

"""
    
    for i, example in enumerate(examples, 1):
        prompt += f"Example {i}:\n{example}\n\n"
    
    prompt += f"Now predict the intent for this scenario:\n{query}\n\nOutput only the intent number (1-8):"
    
    return prompt

# Note: Prompt creation test moved to after function definitions

In [69]:
def convert_features_to_natural_language(X_row, feature_columns, y_value):
    """
    Convert a single data row to natural language description.
    X_row: numpy array of features (one-hot encoded)
    feature_columns: list of feature column names
    y_value: the intent label
    """
    # Parse the one-hot encoded features back to their meaning
    features_dict = {}
    
    for col_name, value in zip(feature_columns, X_row):
        if value == 1:  # Only consider the features that are active (=1)
            features_dict[col_name] = value
    
    # Extract meaningful information
    user = None
    hour = None
    day_of_week = None
    charging_type = None
    bluetooth_type = None
    semantic_location = None
    
    for feature in features_dict.keys():
        if feature.startswith('user_'):
            user = feature.replace('user_', '').upper()
        elif feature.startswith('hour_'):
            hour = int(feature.replace('hour_', ''))
        elif feature.startswith('day_of_week_'):
            day_name = feature.replace('day_of_week_', '')
            day_of_week = day_name.capitalize()
        elif feature.startswith('Charging_type_'):
            charging_type = feature.replace('Charging_type_', '')
        elif feature.startswith('bluetooth_type_'):
            bluetooth_type = feature.replace('bluetooth_type_', '')
        elif feature.startswith('semantic_location_'):
            semantic_location = feature.replace('semantic_location_', '')
    
    # Create natural language description
    description = f"Hour of day: {hour}, "
    description += f"Day: {day_of_week}, "
    description += f"Charging type: {charging_type}, "
    description += f"Bluetooth: {bluetooth_type},"
    description += f"Location: {semantic_location}."    
    intent_name = intent_mapping.get(int(y_value), "unknown")
    description += f"Expected intent: {intent_name}."
    
    return description

# Test the function
feature_cols = onehot_features.columns.tolist()
print("Feature columns (first 10):", feature_cols[:10])
print("\nSample natural language conversion:")
print(convert_features_to_natural_language(X_train[0], feature_cols, y_train[0]))

Feature columns (first 10): ['user_user_1', 'user_user_10', 'user_user_11', 'user_user_12', 'user_user_13', 'user_user_14', 'user_user_15', 'user_user_16', 'user_user_18', 'user_user_19']

Sample natural language conversion:
Hour of day: 12, Day: 4, Charging type: 1, Bluetooth: 1,Location: home.Expected intent: leisure.


In [70]:
# Extract 5 random samples from X_train for few-shot learning
random_indices = random.sample(range(len(X_train)), 5)
few_shot_examples = []
feature_cols = onehot_features.columns.tolist()

for idx in random_indices:
    example = convert_features_to_natural_language(X_train[idx], feature_cols, y_train[idx])
    few_shot_examples.append(example)

print("=" * 80)
print("FEW-SHOT LEARNING EXAMPLES (5 random samples from X_train)")
print("=" * 80)
for i, example in enumerate(few_shot_examples, 1):
    print(f"\nExample {i}:")
    print(example)

FEW-SHOT LEARNING EXAMPLES (5 random samples from X_train)

Example 1:
Hour of day: 23, Day: 3, Charging type: 1, Bluetooth: 1,Location: other.Expected intent: other.

Example 2:
Hour of day: 17, Day: 4, Charging type: 1, Bluetooth: 2,Location: other.Expected intent: other.

Example 3:
Hour of day: 16, Day: 2, Charging type: 1, Bluetooth: 1,Location: other.Expected intent: other.

Example 4:
Hour of day: 18, Day: 1, Charging type: 1, Bluetooth: 2,Location: home.Expected intent: leisure.

Example 5:
Hour of day: 16, Day: 4, Charging type: 1, Bluetooth: 1: Device C1nected,Location: home.Expected intent: leisure.


In [71]:
def extract_intent_from_response(response_text):
    """
    Extract the intent number from Ollama's response.
    The model should output only a number, but we'll try to be robust.
    """
    # Clean the response
    response_clean = response_text.strip()
    
    # Try to find a single digit number (1-8)
    for char in response_clean:
        if char in ['1', '2', '3', '4', '5', '6', '7', '8']:
            return int(char)
    
    # If no valid number found, return -1 (invalid)
    return -1

# Test with a few examples from X_test
print("=" * 80)
print("TESTING OLLAMA MODEL WITH A FEW EXAMPLES")
print("=" * 80)

test_indices = range(min(3, len(X_test)))  # Test with first 3 examples
test_results = []

for test_idx in test_indices:
    query = convert_features_to_natural_language(X_test[test_idx], feature_cols, y_test[test_idx])
    query = query.replace("Expected intent: ", "Predict the intent: ")
    
    prompt = create_few_shot_prompt(few_shot_examples, query)
    
    try:
        # Call Ollama model - adjust model name if needed
        response = ollama.generate(model="gemma3:4b", prompt=prompt, stream=False)
        predicted_intent = extract_intent_from_response(response['response'])
        actual_intent = int(y_test[test_idx])
        
        print(f"\nTest {test_idx + 1}:")
        print(f"  Actual intent: {actual_intent} ({intent_mapping.get(actual_intent, 'unknown')})")
        print(f"  Predicted intent: {predicted_intent} ({intent_mapping.get(predicted_intent, 'unknown')})")
        print(f"  Ollama response: {response['response'].strip()[:100]}...")
        print(f"  Match: {'✓' if predicted_intent == actual_intent else '✗'}")
        
        test_results.append({
            'actual': actual_intent,
            'predicted': predicted_intent,
            'correct': predicted_intent == actual_intent
        })
    except Exception as e:
        print(f"\nTest {test_idx + 1}: Error - {str(e)}")
        print("Make sure Ollama server is running and some model is available.")

TESTING OLLAMA MODEL WITH A FEW EXAMPLES

Test 1:
  Actual intent: 1 (leisure)
  Predicted intent: 1 (leisure)
  Ollama response: 1...
  Match: ✓

Test 2:
  Actual intent: 1 (leisure)
  Predicted intent: 1 (leisure)
  Ollama response: 1...
  Match: ✓

Test 3:
  Actual intent: 1 (leisure)
  Predicted intent: 1 (leisure)
  Ollama response: 1...
  Match: ✓


In [72]:
print("\n" + "=" * 80)
print("FULL BATCH PREDICTION ON ALL X_TEST SAMPLES")
print("=" * 80)

all_predictions = []
all_actuals = []
errors = 0
invalid_responses = 0

print(f"Processing {len(X_test)} test samples...")

for idx in range(len(X_test)):
    if (idx + 1) % 50 == 0:
        print(f"  Progress: {idx + 1}/{len(X_test)}")
    
    query = convert_features_to_natural_language(X_test[idx], feature_cols, y_test[idx])
    query = query.replace("Expected intent: ", "Predict the intent: ")
    
    prompt = create_few_shot_prompt(few_shot_examples, query)
    
    try:
        response = ollama.generate(model="gemma3:4b", prompt=prompt, stream=False)
        predicted_intent = extract_intent_from_response(response['response'])
        actual_intent = int(y_test[idx])
        
        if predicted_intent == -1:
            invalid_responses += 1
        
        all_predictions.append(predicted_intent)
        all_actuals.append(actual_intent)
    except Exception as e:
        print(f"Error at index {idx}: {str(e)}")
        errors += 1
        all_predictions.append(-1)
        all_actuals.append(int(y_test[idx]))

print(f"\nProcessing complete!")
print(f"Total samples: {len(X_test)}")
print(f"Errors encountered: {errors}")
print(f"Invalid responses (couldn't extract number): {invalid_responses}")


FULL BATCH PREDICTION ON ALL X_TEST SAMPLES
Processing 27637 test samples...
  Progress: 50/27637
  Progress: 100/27637
  Progress: 150/27637
  Progress: 200/27637
  Progress: 250/27637
  Progress: 300/27637
  Progress: 350/27637
  Progress: 400/27637
  Progress: 450/27637
  Progress: 500/27637
  Progress: 550/27637
  Progress: 600/27637
  Progress: 650/27637
  Progress: 700/27637
  Progress: 750/27637
  Progress: 800/27637
  Progress: 850/27637
  Progress: 900/27637
  Progress: 950/27637
  Progress: 1000/27637
  Progress: 1050/27637
  Progress: 1100/27637
  Progress: 1150/27637
  Progress: 1200/27637
  Progress: 1250/27637
  Progress: 1300/27637
  Progress: 1350/27637
  Progress: 1400/27637
  Progress: 1450/27637
  Progress: 1500/27637
  Progress: 1550/27637
  Progress: 1600/27637
  Progress: 1650/27637
  Progress: 1700/27637
  Progress: 1750/27637
  Progress: 1800/27637
  Progress: 1850/27637
  Progress: 1900/27637
  Progress: 1950/27637
  Progress: 2000/27637
  Progress: 2050/27637

In [73]:
# Calculate accuracy metrics
all_predictions_array = np.array(all_predictions)
all_actuals_array = np.array(all_actuals)

# Filter out invalid predictions (-1) for accuracy calculation
valid_mask = all_predictions_array != -1
valid_predictions = all_predictions_array[valid_mask]
valid_actuals = all_actuals_array[valid_mask]

if len(valid_predictions) > 0:
    accuracy = (valid_predictions == valid_actuals).sum() / len(valid_predictions)
else:
    accuracy = 0

# Calculate per-class accuracy
print("\n" + "=" * 80)
print("ACCURACY METRICS")
print("=" * 80)
print(f"\nOverall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Valid predictions: {len(valid_predictions)}/{len(all_predictions)}")
print(f"Invalid predictions: {np.sum(all_predictions_array == -1)}")

# Per-class accuracy
print("\nPer-Intent Accuracy:")
print("-" * 40)
for intent_id in range(1, 9):
    class_mask = valid_actuals == intent_id
    if class_mask.sum() > 0:
        class_accuracy = (valid_predictions[class_mask] == intent_id).sum() / class_mask.sum()
        class_count = class_mask.sum()
        print(f"{intent_mapping[intent_id]:12} (ID {intent_id}): {class_accuracy:.4f} ({class_accuracy*100:.2f}%) - {class_count} samples")
    else:
        print(f"{intent_mapping[intent_id]:12} (ID {intent_id}): No samples in test set")

# Confusion Matrix
print("\n" + "=" * 80)
print("CONFUSION MATRIX (rows=actual, cols=predicted)")
print("=" * 80)
confusion_matrix = np.zeros((9, 9), dtype=int)  # 1-8 + 1 for invalid
for actual, predicted in zip(valid_actuals, valid_predictions):
    confusion_matrix[int(actual), int(predicted)] += 1

# Print confusion matrix with labels
print("\n        Predicted Intent")
print("     ", end="")
for i in range(1, 9):
    print(f"{i:6}", end="")
print()
for i in range(1, 9):
    print(f"  {i}  ", end="")
    for j in range(1, 9):
        print(f"{confusion_matrix[i, j]:6}", end="")
    print()

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Model: Ollama (gemma:2b)")
print(f"Method: Few-shot learning with 5 examples")
print(f"Total Test Samples: {len(X_test)}")
print(f"Successful Predictions: {len(valid_predictions)}")
print(f"Final Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


ACCURACY METRICS

Overall Accuracy: 0.9961 (99.61%)
Valid predictions: 27206/27637
Invalid predictions: 431

Per-Intent Accuracy:
----------------------------------------
leisure      (ID 1): 0.9944 (99.44%) - 7024 samples
study        (ID 2): 1.0000 (100.00%) - 1339 samples
work         (ID 3): 1.0000 (100.00%) - 88 samples
travel       (ID 4): No samples in test set
shopping     (ID 5): 1.0000 (100.00%) - 2713 samples
praying      (ID 6): 1.0000 (100.00%) - 574 samples
wellness     (ID 7): 1.0000 (100.00%) - 59 samples
other        (ID 8): 0.9956 (99.56%) - 15409 samples

CONFUSION MATRIX (rows=actual, cols=predicted)

        Predicted Intent
          1     2     3     4     5     6     7     8
  1    6985     0     0     0    34     0     0     5
  2       0  1339     0     0     0     0     0     0
  3       0     0    88     0     0     0     0     0
  4       0     0     0     0     0     0     0     0
  5       0     0     0     0  2713     0     0     0
  6       0     0    

In [74]:
import ollama
import random

# Create mapping for intent numbers to human-readable names
intent_mapping = {
    1: "leisure",
    2: "study",
    3: "work",
    4: "travel",
    5: "shopping",
    6: "praying",
    7: "wellness",
    8: "other"
}

print("Ollama package imported successfully")

Ollama package imported successfully


## Ollama Few-Shot Learning for Intent Prediction

Using local Ollama server with few-shot learning examples to predict user intents.